# 04 · CD: overlap-save y alcance del método

**Objetivo.** Mostrar y validar el modelo por bloques de la dispersión cromática, que es el algoritmo de la FPGA (pasos B3 y B4 de la guía):

- el **overlap-save** con FFT de $L = 2N$ y el filtro $\bar H$ (Ec. 28), su retardo de $N/2$ y la elección automática de $N$;
- el **alcance** de cada $N$: error contra la referencia ideal en función de la distancia (Fig. 9 del PDF);
- el **residuo de β₃** que queda cuando el compensador solo invierte β₂ (Fig. 10 y Tabla 17 del PDF), que justifica incluir β₃ (decisión 4);
- las implicancias para el hardware, con los parámetros provisorios.

Todo sale de `emulador/cd/` (`overlap_save.py`, `reference.py`, `characterize.py`) y del bloque `emulador/blocks/chromatic_dispersion.py`, el mismo código que usan los tests (`tests/test_chromatic_dispersion.py`). Referencias: `referencia_guia_ISI_CD.md` B.4 y B.8; PDF *Fundamentos* §4.7.2, §6.2 y §7 (Ec. 27 a 31, Figs. 8 a 10, Tabla 17).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from emulador import plots
from emulador.blocks import ChromaticDispersion
from emulador.cd import beta3_residual, cd_N, cd_R, cd_hbar, cd_overlap_save, cd_reference
from emulador.config import D_si, HardwareParams, S_si, Scenario
from emulador.metrics import nrmse
from emulador.stimuli import comm_signal

plots.use_style()

## 1. Teoría

**Muestreo en frecuencia.** Se evalúa $H_{CD}$ en las frecuencias de una DFT de $L = 2N$ puntos y se le agrega una fase lineal que centra el chirp (no causal) en la ventana:

$$
\bar H[k] = H_{CD}(2\pi f_k)\,e^{-j2\pi k (N/2)/L}, \qquad f_k = k\,\frac{f_s}{L}.
$$

**Overlap-save** (Fig. 8 del PDF). Por cada bloque de $N$ muestras nuevas:

```
ventana de L = 2N:  [ N muestras anteriores | N muestras nuevas ]
                    -> FFT -> x H̄[k] -> IFFT -> descartar las primeras N -> N muestras de salida
```

El primer bloque va precedido por $N$ ceros. La salida queda **retrasada $N/2$ muestras** (retardo puro, que el bloque declara en `delay_samples`). $\bar H$ lo calcula el host una vez por configuración; la FPGA solo hace FFT, producto e IFFT.

**Condición de largo.** El muestreo en frecuencia periodiza la respuesta al impulso: si el chirp ($R$ muestras, Ec. 26) es más largo que lo que se descarta, sus colas se superponen (aliasing temporal). Hace falta $N \ge R$ (decisión 5); `cd_N` elige la menor potencia de 2 que cumple, con un mínimo de 128 y un máximo $N_{max}$ (hardware).

## 2. Parámetros

In [ ]:
esc = Scenario()        # 32 GBd, 2 sps -> fs = 64 GSa/s
fs, lam = esc.fs, esc.wavelength
D = D_si(17.0)
S = S_si(0.057)
KM = 1e3
N_SYM = 10_000
SEED = 1
hw = HardwareParams()

print(esc.to_dict())
print(hw.to_dict(), "<- PROVISORIOS (decisión 7)")

## 3. Retardo N/2 y elección automática de N

Un impulso en $n_0$ sale del overlap-save crudo (`process_hw`) como el chirp centrado en $n_0 + N/2$; la salida alineada (`process`) lo deja en $n_0$.

In [ ]:
z_demo = 100 * KM
blk = ChromaticDispersion(length_km=100.0)
n0 = 300
x_imp = np.zeros(1024, complex)
x_imp[n0] = 1.0
y_hw = blk.process_hw(x_imp)
y_al = blk(x_imp)

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(np.abs(y_hw), label=f"process_hw (crudo): centro en n0 + N/2 = {n0 + blk.N // 2}")
ax.plot(np.abs(y_al), label=f"process (alineado): centro en n0 = {n0}")
ax.axvline(n0, ls=":", color="C1")
ax.axvline(n0 + blk.N // 2, ls=":", color="C0")
ax.set_xlim(n0 - 100, n0 + blk.N // 2 + 100)
ax.set_xlabel("n")
ax.set_title(f"|y[n]| para un impulso, 100 km, N = {blk.N} (R = {cd_R(fs, D, lam, z_demo)})")
ax.legend()
plots.save_fig(fig, "04_retardo_N2")

# el chirp tiene módulo casi constante: el centro se mide con el centro de masa de |y|²
centroid = lambda y: float(np.sum(np.arange(y.size) * np.abs(y) ** 2) / np.sum(np.abs(y) ** 2))  # noqa: E731
center_hw, center_al = centroid(y_hw), centroid(y_al)
print(f"centro de |y|²: crudo {center_hw:.2f}, alineado {center_al:.2f}")
print("N =", blk.N, " delay_samples =", blk.delay_samples)
print("z [km]        10    100    229    230    500   1000   1500   2000")
print("R      " + "".join(f"{cd_R(fs, D, lam, z * KM):7d}" for z in (10, 100, 229, 230, 500, 1000, 1500, 2000)))
print("N      " + "".join(f"{cd_N(fs, D, lam, z * KM):7d}" for z in (10, 100, 229, 230, 500, 1000, 1500, 2000)))

**Qué muestra la figura.** Un impulso en $n_0$ = 300 sale del overlap-save crudo (`process_hw`) como el chirp de 100 km centrado en $n_0 + N/2$ = 364 (centro de masa medido: 364,01), con un ancho de unas R = 55 muestras. La salida alineada (`process`) lo deja centrado en $n_0$ = 300. Es el retardo $N/2$ que el bloque declara en `delay_samples` y que el RTL va a tener.

## 4. Alcance del método (Fig. 9 del PDF)

NRMSE del overlap-save contra la referencia ideal, con QAM16 a 32 GBd y 2 sps, descontando el retardo $N/2$ y los bordes. La línea punteada es NRMSE = 1e-2 (criterio de la decisión 21).

In [ ]:
x = comm_signal(esc, mod="qam16", n_symbols=N_SYM, seed=SEED).samples
Z = np.arange(25, 2001, 25)
NS = (128, 256, 512, 1024, 2048)
err = {N: [] for N in NS}
err_auto = []
for z_km in Z:
    z = z_km * KM
    r = cd_reference(x, fs, D, lam, z, S=S)
    for N in NS:
        y = cd_overlap_save(x, cd_hbar(N, fs, D, lam, z, S=S))
        m = x.size - 2 * N
        err[N].append(nrmse(y[N // 2 : N // 2 + m], r[:m]))
    err_auto.append(err[min(cd_N(fs, D, lam, z, N_max=np.inf), 2048)][-1])

fig, ax = plt.subplots(figsize=(11, 5))
for N in NS:
    ax.semilogy(Z, err[N], label=f"OLS, N = {N}")
ax.axhline(1e-2, ls=":", color="k")
ax.set_ylim(1e-5, 2)
ax.set_xlabel("z [km]")
ax.set_ylabel("NRMSE contra la referencia")
ax.set_title("QAM16, 32 GBd, 2 sps, fs = 64 GSa/s")
ax.legend(ncol=2)
plots.save_fig(fig, "04_alcance_ols")

reach = {}  # último z antes de que el NRMSE cruce 1e-2 (Z[-1] si nunca cruza)
for N in NS:
    above = np.nonzero(np.array(err[N]) >= 1e-2)[0]
    reach[N] = int(Z[above[0] - 1]) if above.size and above[0] > 0 else (0 if above.size else int(Z[-1]))
print(" N      alcance (NRMSE < 1e-2)   R máximo que entra")
for N in NS:
    print(f"{N:5d}   {reach[N]:8d} km{'' if reach[N] < Z[-1] else '+':1s}             {N}")
print(f"NRMSE con N = cd_N: máximo {max(err_auto):.1e} (z = {Z[int(np.argmax(err_auto))]} km)")

**Qué muestra la figura.** Cada curva del overlap-save sube de golpe cuando el largo del chirp R supera el bloque N: el NRMSE cruza 1e-2 a 225 / 475 / 1000 km con N = 128 / 256 / 512, y con N = 1024 y 2048 queda por debajo hasta 2000 km. Con el N automático (N ≥ R) el peor error es 8,8e-3. Un FIR en el tiempo necesitaría R taps (55 a 100 km, 1117 a 2000 km), muy por encima de los 32 del FIR de la ISI: la CD va en frecuencia (decisión 5).

## 5. Residuo de β₃ tras compensar solo β₂ (Fig. 10 y Tabla 17 del PDF, decisión 23)

Canal con β₂ y β₃ (SSMF, S = 0,057 ps/nm²/km), seguido de una compensación que solo invierte β₂ ($-D$, $S = 0$), sin ecualizador adaptativo. Se usa la referencia ideal (a 200 GBd y 1000 km, $R$ ≈ 22 000). El residuo se expresa como SNR equivalente, $-20\log_{10}(\mathrm{EVM})$.

β₃ agrega además un **retardo global con parte fraccionaria**. Se mide de dos formas: con la **grilla de 2 sps** (el receptor solo elige fases enteras, como la Tabla 17 del PDF) y con **temporización ideal** (además, el mejor retardo fraccionario continuo). La diferencia es error de temporización del receptor, no ISI de β₃.

In [ ]:
RATES = (32e9, 64e9, 100e9, 200e9)
res_grid = {Rs: beta3_residual(Scenario(Rs=Rs), D, 1000 * KM, S) for Rs in RATES}
res_ideal = {Rs: beta3_residual(Scenario(Rs=Rs), D, 1000 * KM, S, timing="ideal") for Rs in RATES}
control = {Rs: beta3_residual(Scenario(Rs=Rs), D, 1000 * KM, 0.0).evm for Rs in (32e9, 100e9)}

fig, axs = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, Rs in zip(axs[:3], (32e9, 100e9, 200e9)):
    r = res_ideal[Rs]
    plots.constellation(ax, r.symbols, ref=r.ref, title=f"{Rs / 1e9:.0f} GBd, 1000 km · EVM {100 * r.evm:.1f} %")
rates_curve = np.array([32, 50, 64, 80, 100, 125, 150, 175, 200]) * 1e9
snr_curve = {}
for i, z_km in enumerate((500, 1000, 2000)):
    for timing, ls in (("ideal", "o-"), ("grid", "x:")):
        snr_curve[z_km, timing] = [-20 * np.log10(beta3_residual(Scenario(Rs=Rs), D, z_km * KM, S, timing=timing).evm)
                                   for Rs in rates_curve]
        axs[3].plot(rates_curve / 1e9, snr_curve[z_km, timing], ls, ms=4, color=f"C{i}",
                    label=f"{z_km} km" + (" (grilla 2 sps)" if timing == "grid" else ""))
axs[3].axhline(20, ls="--", color="k", lw=0.8, label="20 dB (ref.)")
axs[3].set_xlabel("Tasa de símbolos [GBd]")
axs[3].set_title("SNR equiv. del residuo de β₃ [dB]")
axs[3].legend(fontsize=8)
plots.save_fig(fig, "04_residuo_beta3")

print("Tasa (1000 km)   EVM grilla 2 sps   EVM temporiz. ideal   SNR eq. ideal   PDF (Tabla 17)")
for Rs, pdf in zip(RATES, ("0,7 %", "5,6 %", "22 %", ">100 %")):
    eg, ei = res_grid[Rs].evm, res_ideal[Rs].evm
    print(f"{Rs / 1e9:6.0f} GBd     {100 * eg:13.2f} %   {100 * ei:16.2f} %   {-20 * np.log10(ei):10.1f} dB   {pdf:>10s}")
print("control sin β₃:", ", ".join(f"{Rs / 1e9:.0f} GBd {100 * e:.2f} %" for Rs, e in control.items()))

**Qué muestra la figura.**
- **Constelaciones** (1000 km, temporización ideal): a 32 GBd el residuo de β₃ es despreciable (0,3 %), a 100 GBd deja nubes visibles (8,3 %) y a 200 GBd la constelación queda destruida (69 %).
- **Curvas continuas** (temporización ideal): el residuo crece de forma suave con la tasa y la distancia; cruza 20 dB de SNR equivalente cerca de 135 / 107 / 85 GBd a 500 / 1000 / 2000 km. Para el Proyecto Marco (≥ 100 GBd) el ecualizador tiene que absorberlo: es el motivo de incluir β₃ (decisión 4).
- **Curvas punteadas** (grilla de 2 sps, como la Tabla 17 del PDF): quedan por debajo y suben y bajan de forma errática, porque mezclan la ISI con el retardo global fraccionario que agrega β₃ y que ese receptor no corrige (decisión 23).

## 6. Implicancias para el hardware (Ref. B.8)

**Parámetros provisorios** (decisión 7): el RFSoC no está confirmado y $f_{clk}$ no está definida. La tasa efectiva del bloque con overlap-save es $R_{CD} = P\cdot f_{clk}\cdot N/L = P\cdot f_{clk}/2$ (Ec. 31). La tabla de $N$ necesario usa `cd_N` sin tope y marca los casos que no entran en $N_{max}$ = 2048.

In [ ]:
F_CLK_EJEMPLO = 400e6  # ilustrativo: NO decidido
print(f"Tasa efectiva con f_clk = {F_CLK_EJEMPLO / 1e6:.0f} MHz (ejemplo, no decidido):")
for P in (1, 8, 64, 128, 256, 320):
    r_cd = P * F_CLK_EJEMPLO / 2
    print(f"  P = {P:4d}: R_CD = {r_cd / 1e9:7.2f} GSa/s ({100 * r_cd / fs:6.1f} % de fs = {fs / 1e9:.0f} GSa/s)")

print(f"\nN necesario (2 sps, SSMF), N_max = {hw.N_max} (provisorio); * = no entra")
Z_HW = (100, 500, 1000, 2000)
print("Rs [GBd] " + "".join(f"{z:>9d} km" for z in Z_HW))
n_table = {}
for Rs in RATES:
    esc_r = Scenario(Rs=Rs)
    row = ""
    for z_km in Z_HW:
        N = cd_N(esc_r.fs, D, lam, z_km * KM, N_max=np.inf)
        n_table[Rs, z_km] = N
        row += f"{N:>10d}{'*' if N > hw.N_max else ' ':1s}  "
    print(f"{Rs / 1e9:8.0f} {row}")

## 7. Validación

Los mismos criterios que `tests/test_chromatic_dispersion.py`.

In [ ]:
assert blk.delay_samples == blk.N // 2 == 64
assert abs(center_hw - (n0 + blk.N // 2)) < 0.5 and abs(center_al - n0) < 0.5
# alcance medido (NRMSE < 1e-2): 225 / 475 / 1000 / > 2000 km; el PDF da ~200 / 300-400 / 800-900 / ~1500
assert 200 <= reach[128] < 300 and 450 <= reach[256] < 600 and 900 <= reach[512] < 1200 and reach[1024] >= 1800
assert all(cd_R(fs, D, lam, reach[N] * KM) <= 1.15 * N for N in (128, 256, 512))  # regla N ≳ R
assert max(err_auto) < 1.5e-2
for Rs, ref in ((32e9, 0.007), (64e9, 0.056), (100e9, 0.22)):
    assert abs(res_grid[Rs].evm / ref - 1) < 0.05
for Rs, ref in ((32e9, 0.0029), (64e9, 0.022), (100e9, 0.083)):
    assert abs(res_ideal[Rs].evm / ref - 1) < 0.10
assert res_grid[200e9].evm > 0.5 and res_ideal[200e9].evm > 0.5
for z_km in (500, 1000, 2000):
    assert all(a > b for a, b in zip(snr_curve[z_km, "ideal"], snr_curve[z_km, "ideal"][1:]))
assert all(e < 0.002 for e in control.values())
assert n_table[32e9, 2000] == 2048 and n_table[200e9, 1000] > hw.N_max
print("OK")

## 8. Conclusiones

- **Overlap-save**: con $L = 2N$ y $\bar H$ de la Ec. 28, la salida cruda sale retrasada exactamente $N/2$ (el centro de un impulso pasa de $n_0$ a $n_0 + N/2$) y `process` lo descuenta. $\bar H$ se calcula una vez; la FPGA solo hace FFT, producto e IFFT.
- **Alcance** (Fig. 9, QAM16 a 32 GBd): el error crece bruscamente cuando $R$ supera $N$. NRMSE < 1e-2 hasta 225 km con N = 128, 475 km con 256, 1000 km con 512 y más de 2000 km con 1024. Son algo mayores que los del PDF (~200 / 300–400 / 800–900 / ~1500 km), porque la QAM16 casi no tiene energía cerca de Nyquist, donde están las colas del chirp que no entran (decisión 21); con ruido blanco el margen desaparece. La regla $N \ge R$ de `cd_N` es la segura: con ella el peor NRMSE es 8,8e-3.
- **Por qué en frecuencia**: un FIR que emule la CD en el tiempo necesitaría R taps (55 a 100 km, 1117 a 2000 km), muy por encima de los 32 del FIR de la ISI. El overlap-save cubre los 2000 km con N = 2048 (decisión 5).
- **Residuo de β₃** (decisión 23): tras compensar solo β₂, a 1000 km y con temporización ideal la EVM es 0,29 % a 32 GBd (despreciable), 2,2 % a 64 GBd, 8,3 % a 100 GBd (SNR equivalente 21,6 dB, comparable al ruido del enlace) y 69 % a 200 GBd (constelación destruida); crece monótonamente con la tasa y la distancia. El control sin β₃ queda en el piso del RRC (0,12 %). Sin β₃ en el emulador, esa ISI residual nunca aparecería en las pruebas de la cadena de DSP: es el motivo de la decisión 4.
- **Temporización**: con la grilla de 2 sps (como la Tabla 17 del PDF) la EVM a 32, 64 y 100 GBd es 0,69 %, 5,5 % y 21 %, porque β₃ también agrega un retardo global fraccionario que un receptor sin interpolación no corrige; la curva deja de ser monótona. La Tabla 17 mezcla ese error de temporización con la ISI de β₃: conviene corregirla en el informe y avisar a la tesis de CDC/ecualización.
- **Hardware** (parámetros provisorios): con $N_{max}$ = 2048, a 32 GBd entran los 2000 km; a 64 GBd solo hasta ~500 km, y a 100–200 GBd haría falta $N$ de 4096 a 65 536. La tasa efectiva $R_{CD} = P f_{clk}/2$ exige $P$ = 320 para seguir los 64 GSa/s con $f_{clk}$ = 400 MHz (ejemplo). $N_{max}$, $P$ y $f_{clk}$ se cierran en C1.

Figuras en `docs/golden_model/figuras/04_*.png`.